In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [25]:
df = pd.read_csv('../data/data.csv')

In [31]:
def process_saving_data():
    # Create a copy of the dataframe to avoid modifying the original
    processed_df = df.copy()
    
    expense_cols = [
        'Rent', 'Loan_Repayment', 'Insurance', 'Groceries', 'Transport', 
        'Eating_Out', 'Entertainment', 'Utilities', 'Healthcare', 
        'Education', 'Miscellaneous'
    ]

    processed_df["total_expenses"] = processed_df[expense_cols].sum(axis=1)

    processed_df['actual_savings'] = processed_df['Income'] - processed_df['total_expenses']
    processed_df['savings_diff'] = abs(processed_df['actual_savings'] - processed_df['Disposable_Income'])

    processed_df['savings_ratio'] = processed_df['actual_savings'] / processed_df['Income']
    # Add this line to calculate spend_ratio
    processed_df['spend_ratio'] = processed_df['total_expenses'] / processed_df['Income']

    potential_savings_cols = [col for col in processed_df.columns if col.startswith('Potential_Savings_')]
    processed_df['total_potential_savings'] = processed_df[potential_savings_cols].sum(axis=1)    

    processed_df['goal_achieved'] = np.where(processed_df['actual_savings'] >= processed_df['Desired_Savings'], 1, 0)

    categorical_cols = ['Occupation', 'City_Tier']
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        processed_df[col + '_encoded'] = le.fit_transform(processed_df[col].astype(str))
        label_encoders[col] = le

    numerical_cols = ['Income', 'Age', 'Dependents'] + expense_cols + [
        'actual_savings', 'savings_ratio', 'spend_ratio', 'total_potential_savings'
    ]

    scaler = StandardScaler()
    processed_df[numerical_cols] = scaler.fit_transform(processed_df[numerical_cols])
    
    # Drop any rows with missing values
    processed_df = processed_df.dropna()

    categorical_cols = ['Occupation', 'City_Tier']
    processed_df = processed_df.drop(columns=categorical_cols)
    
    return processed_df, scaler, label_encoders

In [32]:
df_processed, scaler, label_encoders = process_saving_data()

# Display some information about the processed dataframe
print(f"Processed dataframe shape: {df_processed.shape}")
print("\nFirst 5 rows of processed data:")
display(df_processed.head())
print(scaler)
print(label_encoders)

# Show some statistics
print("\nSummary statistics for key metrics:")
print(f"Average savings ratio: {df_processed['savings_ratio'].mean():.4f}")
print(f"Percentage of people achieving savings goals: {df_processed['goal_achieved'].mean()*100:.2f}%")

Processed dataframe shape: (20000, 34)

First 5 rows of processed data:


,Income,Age,Dependents,Rent,Loan_Repayment,Insurance,Groceries,Transport,Eating_Out,Entertainment,Utilities,Healthcare,Education,Miscellaneous,Desired_Savings_Percentage,Desired_Savings,Disposable_Income,Potential_Savings_Groceries,Potential_Savings_Transport,Potential_Savings_Eating_Out,Potential_Savings_Entertainment,Potential_Savings_Utilities,Potential_Savings_Healthcare,Potential_Savings_Education,Potential_Savings_Miscellaneous,total_expenses,actual_savings,savings_diff,savings_ratio,spend_ratio,total_potential_savings,goal_achieved,Occupation_encoded,City_Tier_encoded
0,0.076268,0.586855,-1.407998,0.462036,-0.478737,0.503356,0.288553,-0.025315,0.128200,0.058651,0.161248,-0.071350,-0.806345,0.001150,13.890948,6200.537192,11265.627707,1685.696222,328.895281,465.769172,195.151320,678.292859,67.682471,0.000000,85.735517,33371.621929,0.052661,1.818989e-12,-0.028192,0.028192,0.360089,1,2,0
1,-0.368048,-0.517841,0.002857,-0.404558,-0.478737,-0.392194,-0.474048,-0.435606,-0.548371,-0.267708,-0.348340,-0.322906,-0.305525,-0.315170,7.160376,1923.176434,9676.818733,540.306561,119.347139,141.866089,234.131168,286.668408,6.603212,56.306874,97.388606,17181.777859,-0.082668,1.818989e-12,1.082096,-1.082096,-0.425674,1,1,1
2,0.219478,-0.444195,-0.702570,-0.168614,0.598434,0.500215,0.219935,0.193877,0.035068,0.184323,0.342254,0.316582,0.213557,-0.239269,13.997808,7050.360422,13891.450624,1466.073984,473.549752,410.857129,459.965256,488.383423,7.290892,106.653597,138.542422,36476.154459,0.276319,0.000000e+00,0.212781,-0.212781,0.377202,1,3,2
3,1.496246,-1.475245,-1.407998,0.659482,1.111629,2.300480,1.883401,1.650864,2.415183,0.946513,1.436262,1.786129,-0.806345,2.006515,16.455440,16694.965136,31617.953615,1875.932770,762.020789,1241.017448,320.190594,1389.815033,193.502754,0.000000,296.041183,69837.646632,1.786199,3.637979e-12,0.581574,-0.581574,1.358027,1,2,2
4,-0.417614,0.807795,1.413712,-0.447422,0.248222,-0.548678,-0.431178,-0.535695,-0.519045,-0.529668,-0.559783,-0.303408,-0.339900,-0.372832,7.533982,1874.099434,6265.700532,788.953124,68.160766,61.712505,187.173750,194.117130,47.294591,67.388120,96.557076,18609.583016,-0.373215,0.000000e+00,-0.033308,0.033308,-0.414520,1,0,1


StandardScaler()
{'Occupation': LabelEncoder(), 'City_Tier': LabelEncoder()}

Summary statistics for key metrics:
Average savings ratio: 0.0000
Percentage of people achieving savings goals: 98.06%


In [ ]:
df_processed.to_csv('../data/processed_data.csv', index=False)

# Save the fitted scaler and label encoders for later use
with open('../model/preprocessing_objects.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'label_encoders': label_encoders}, f)

print("Preprocessing objects saved successfully!")